In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from functools import partial

import time
from tqdm.notebook import tqdm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}"

In [ ]:
import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax

import gpjax as gpx

In [ ]:
cols = mpl.rcParams["axes.prop_cycle"].by_key()["color"]

---

In [ ]:
from dmpe.utils.density_estimation import build_grid, DensityEstimate
from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

Instead of optimizing the GP posterior uncertainty along a trajectory, utilize a gridded version of the GP posterior, at which the loss is evaluated.

- what must be changed? Only the cost function?

In [ ]:
from dmpe.algorithms.algorithm_utils import default_dmpe_parameterization
from dmpe.models.models import NeuralEulerODEPendulum, NeuralEulerODE

from dmpe.excitation.excitation_utils import soft_penalty

from dmpe.evaluation.plotting_utils import plot_feature_combinations, plot_sequence

In [ ]:
from admpe.gp_excitation.algorithm import excite_with_gp_uncertainty

In [ ]:
env, penalty_function, featurize, env_params = setup_fluid_tank_env()

penalty_function = lambda x, u: soft_penalty(a=x, a_max=1, penalty_order=2) * 1e3 + soft_penalty(
    a=u, a_max=1, penalty_order=2
) * 1e3

exp_params, proposed_actions, loader_key, expl_key, process_noise_key = default_dmpe_parameterization(
    env, seed=0, n_time_steps=5000, featurize=featurize, model_class=NeuralEulerODE,
)

In [ ]:
exp_params["alg_params"]["loss_type"] = "grid"

exp_params["alg_params"]["target_support_points"] = build_grid(env.action_dim + env.obs_dim, -1.0, 1.0, 20)

In [ ]:
observations, actions, model, gp_model, losses, proposed_actions, callback_out = excite_with_gp_uncertainty(
    env, exp_params, proposed_actions, loader_key, expl_key, process_noise_key
)

In [ ]:
plot_sequence(
    observations[:-1],
    actions,
    env.tau,
    env.obs_description,
    env.action_description,
)

In [ ]:
for posterior in gp_model.posteriors:
    gpx.summarise(posterior)

In [ ]:
plot_sequence(
    observations[1800:2000],
    actions[1800:2000],
    env.tau,
    env.obs_description,
    env.action_description,
)

In [ ]:
plot_feature_combinations(jnp.hstack([
    observations[:-1],
    actions,
]), list(env.obs_description) + list(env.action_description));
plt.show()

In [ ]:
x_plot = build_grid(dim=env.obs_dim + env.action_dim, low=-1.05, high=1.05, points_per_dim=25)
x_plot.shape

In [ ]:
var = gp_model.predict_posterior_variance(x_plot)

In [ ]:
gpx.summarise(gp_model)

In [ ]:
DensityEstimate(p=var[:, 0], z_g=x_plot, bandwidth=0.1, n_observations=0).visualize()